# Notebook 05 — Process Reward Model Toy

**Chapter**: [Chapter 3 — Sampling and Verification](../chapters/03-sampling-and-verification.md).

**Claim demonstrated**: A tiny PRM trained on a synthetic stepwise arithmetic task identifies bad intermediate steps with above-chance accuracy and, used as a reranker, beats an outcome-only ORM at the same labeling budget.

**Hardware**: CPU is fine. ≤ 5 minutes end-to-end.

**Why synthetic**: the point is mechanism, not scale. A real PRM (e.g. Math-Shepherd) takes a few thousand GPU-hours; the *signal* — step-labels carry per-dollar information that trace-labels don't — reproduces on a 1k-example synthetic toy.

---

## Task

Multi-step addition with deliberately corrupted intermediate steps. Each example is a chain like:

```
Step 1: 3 + 4 = 7
Step 2: 7 + 2 = 9
Step 3: 9 + 5 = 14
Final: 14
```

We generate clean chains, then corrupt each step with probability 0.2 by replacing the right-hand side with a wrong number. The label per step: 1 if correct, 0 if corrupted. The trace label: 1 if the *final* answer is correct.

The PRM is a small MLP / small transformer that takes a step (encoded as digits) and predicts its label. The ORM takes the whole trace and predicts the final-answer label.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import List, Tuple

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

N_TRAIN = 1000
N_TEST = 200
CHAIN_LEN = 4
DIGIT_MAX = 9
CORRUPT_P = 0.2

In [ ]:
@dataclass
class Example:
    steps: List[Tuple[int, int, int]]   # (a, b, claimed_sum)
    step_labels: List[int]              # 1 = correct, 0 = corrupted
    trace_label: int                    # 1 if claimed final result is correct

def make_example():
    a = random.randint(0, DIGIT_MAX)
    running = a
    steps = []
    step_labels = []
    for _ in range(CHAIN_LEN):
        b = random.randint(0, DIGIT_MAX)
        true_sum = running + b
        if random.random() < CORRUPT_P:
            # corrupt with a small offset
            offset = random.choice([-2, -1, 1, 2])
            claimed = true_sum + offset
            step_labels.append(0)
        else:
            claimed = true_sum
            step_labels.append(1)
        steps.append((running, b, claimed))
        # Critically, downstream steps continue from the *claimed* result, not the true one.
        running = claimed
    trace_label = int(all(s == 1 for s in step_labels))
    return Example(steps, step_labels, trace_label)

train = [make_example() for _ in range(N_TRAIN)]
test = [make_example() for _ in range(N_TEST)]
print("Sample:", train[0])

In [ ]:
# Encode each step as a feature vector. PRM operates on steps, ORM on traces (concatenated).
def encode_step(s):
    a, b, c = s
    return torch.tensor([a, b, c, a + b, abs(c - (a + b))], dtype=torch.float32)

def encode_trace(steps):
    return torch.cat([encode_step(s) for s in steps])

STEP_DIM = 5
TRACE_DIM = STEP_DIM * CHAIN_LEN

In [ ]:
class PRM(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(STEP_DIM, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

class ORM(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(TRACE_DIM, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_prm(prm, epochs=20):
    opt = torch.optim.Adam(prm.parameters(), lr=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        random.shuffle(train)
        total = 0.0; n = 0
        for ex in train:
            for step, label in zip(ex.steps, ex.step_labels):
                x = encode_step(step); y = torch.tensor(float(label))
                opt.zero_grad()
                logit = prm(x); loss = loss_fn(logit, y); loss.backward(); opt.step()
                total += loss.item(); n += 1
        if (ep + 1) % 5 == 0:
            print(f"PRM ep {ep+1}: loss = {total/n:.4f}")
    return prm

def train_orm(orm, epochs=20):
    opt = torch.optim.Adam(orm.parameters(), lr=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        random.shuffle(train)
        total = 0.0; n = 0
        for ex in train:
            x = encode_trace(ex.steps); y = torch.tensor(float(ex.trace_label))
            opt.zero_grad()
            logit = orm(x); loss = loss_fn(logit, y); loss.backward(); opt.step()
            total += loss.item(); n += 1
        if (ep + 1) % 5 == 0:
            print(f"ORM ep {ep+1}: loss = {total/n:.4f}")
    return orm

prm = train_prm(PRM())
orm = train_orm(ORM())

In [ ]:
# Evaluation: as rerankers.
# Setup: for each test problem, generate 4 candidate traces (resample). Pick the highest-scored.
# The 'correct' trace is one with trace_label == 1.
import torch

N_CAND = 4

def score_prm(trace_steps):
    with torch.no_grad():
        logits = torch.stack([prm(encode_step(s)) for s in trace_steps])
    # Aggregate: minimum (a chain is only as good as its worst step).
    return logits.min().item()

def score_orm(trace_steps):
    with torch.no_grad():
        return orm(encode_trace(trace_steps)).item()

def bon_eval(score_fn):
    hits = 0
    for _ in range(N_TEST):
        cands = [make_example() for _ in range(N_CAND)]
        best = max(cands, key=lambda c: score_fn(c.steps))
        hits += best.trace_label
    return hits / N_TEST

print(f"BoN-{N_CAND} with PRM: {bon_eval(score_prm):.3f}")
print(f"BoN-{N_CAND} with ORM: {bon_eval(score_orm):.3f}")
print(f"Random pick: {(1 - CORRUPT_P)**CHAIN_LEN:.3f}")

## Interpretation

Expected: BoN with PRM noticeably beats BoN with ORM at this scale, with both above the random-pick baseline.

The mechanism: the PRM has *step-level* signal — it can recognize a corrupted step even in a trace whose final answer happens to be right (the ORM is blind to this). With many candidates, picking by min-PRM score is selecting for the chain with the *best worst step*.

**Lessons for real PRMs**:
- Aggregation matters. Min works well here; product, mean, learned aggregators are real alternatives.
- Label noise is a problem: in Math-Shepherd-style synthetic labels, the 'good step' label can be wrong. Filter aggressively.
- The win over ORMs is largest when traces are long and corruption rates are low — the regime that makes ORM-blindness expensive.

## Going further

- Increase `CHAIN_LEN` to 10. The PRM-ORM gap should widen.
- Reduce `N_TRAIN` to 100. The PRM degrades slowly; the ORM degrades fast (fewer labels per unit signal).
- Swap the MLPs for tiny transformers; the qualitative story holds.